In [18]:
import pandas as pd
import numpy as np
import sys
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix

from xgboost import XGBClassifier
from scipy.sparse import hstack

sys.path.append(os.path.abspath(".."))

In [19]:
df = pd.read_csv("../data/raw/credit_dataset.csv")

In [20]:
np.random.seed(42)

location_text = [
    "South Mumbai affluent area",
    "Bandra West residential zone",
    "Lower Parel commercial district",
    "South Delhi posh locality",
    "Gurgaon corporate hub",
    "Noida sector planned housing",
    "Bangalore IT corridor",
    "Whitefield tech park area",
    "Electronic City industrial belt",
    "Hyderabad HITEC City",
    "Pune Hinjewadi IT park",
    "Chennai OMR corridor",
    "Ahmedabad industrial zone",
    "Surat textile business district",
    "Jaipur urban residential area",
    "Indore emerging commercial zone",
    "Kochi port-adjacent locality",
    "Trivandrum government employee area",
    "Coimbatore manufacturing cluster",
    "Nagpur logistics corridor",
    "Tier 2 industrial town",
    "Tier 2 service economy town",
    "Semi rural district",
    "Agricultural dominant region",
    "Mining dependent region",
    "Tourism seasonal economy area",
    "Border trade town",
    "Small town market center",
    "Urban middle income locality",
    "Urban informal settlement"
]

df["location_text"]=np.random.choice(location_text,size=len(df))

In [21]:
officer_notes = [
    "GST returns consistent for last 3 years",
    "GST registration present but filings irregular",
    "Income primarily cash based limited documentation",
    "Bank statements show frequent cash deposits",
    "Borrower hesitant during verification call",
    "Borrower cooperative during verification call",
    "Stable salaried employment verified",
    "Employment contract valid and confirmed",
    "Business income volatile seasonal variation",
    "Business revenue declining over last year",
    "High dependency on single client",
    "Multiple income sources disclosed",
    "Credit card utilisation consistently high",
    "Credit card utilisation within limits",
    "Past loan closed with minor delays",
    "Past loan closed without delays",
    "Address verification partially successful",
    "Address verification fully successful",
    "Guarantor details provided and verified",
    "No guarantor available for loan",
    "Property ownership documents verified",
    "Property documents pending clarification",
    "Borrower frequently changed residences",
    "Borrower residence stable for many years",
    "Self employed professional with stable clients",
    "Self employed trader with irregular turnover",
    "Salary credited through employer account",
    "Income supported by informal family business",
    "High household expenses relative to income",
    "Household expenses within expected range"
]

df["officer_notes"] = np.random.choice(officer_notes, size=len(df))

In [22]:
export_cols = [
    "age",
    "income",
    "debt_to_income",
    "credit_score",
    "loan_amount",
    "location_text",
    "officer_notes",
    "approved"
]

df[export_cols].to_csv(
    "../data/processed/credit_tab_text_random.csv",
    index=False
)


In [23]:
X_tabular = df[[
    "age",
    "income",
    "debt_to_income",
    "credit_score",
    "loan_amount"
]]

y = df["approved"]

In [24]:
X_train_tab, X_test_tab, y_train, y_test = train_test_split(
    X_tabular,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [25]:
tfidf_location = TfidfVectorizer(
    max_features=10,
    stop_words="english"
)

X_train_loc = tfidf_location.fit_transform(
    df.loc[X_train_tab.index, "location_text"]
)
X_test_loc = tfidf_location.transform(
    df.loc[X_test_tab.index, "location_text"]
)


In [26]:
tfidf_notes = TfidfVectorizer(
    max_features=20,
    stop_words="english"
)

X_train_notes = tfidf_notes.fit_transform(
    df.loc[X_train_tab.index, "officer_notes"]
)
X_test_notes = tfidf_notes.transform(
    df.loc[X_test_tab.index, "officer_notes"]
)

In [27]:
X_train = hstack([
    X_train_tab.values,
    X_train_loc,
    X_train_notes
])

X_test = hstack([
    X_test_tab.values,
    X_test_loc,
    X_test_notes
])

In [28]:
cost_fp = 100
cost_fn = 10

sample_weights = np.where(y_train == 0, cost_fp, cost_fn)

In [29]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train, sample_weight=sample_weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [30]:
y_pred = xgb.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

expected_cost = FP * cost_fp + FN * cost_fn
cost_per_applicant = expected_cost / len(y_test)

cm, expected_cost, cost_per_applicant

(array([[1157,   43],
        [ 437, 1363]]),
 8670,
 2.89)

Did text reduce FP, FN, or both?
    BOTH increased.

Which text source helped more: location or notes?
    Cannot answer. Not enough information, how to evaluate? It currently feels like it did not help because both the FP and FN increased. It added noise without signals. 

Did cost improve vs Notebook 08?
    No

Why might noisy text still help?
    It might help with final evaluations given human intervention in the last stage.